In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
df = sns.load_dataset('titanic')
df.head(15)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
5,0,3,male,NaN,0,0,8.4583,Q,Third,man,True,NaN,Queenstown,no,True
6,0,1,male,54.0,0,0,51.8625,S,First,man,True,E,Southampton,no,True
7,0,3,male,2.0,3,1,21.0750,S,Third,child,False,NaN,Southampton,no,False
8,1,3,female,27.0,0,2,11.1333,S,Third,woman,False,NaN,Southampton,yes,False
9,1,2,female,14.0,1,0,30.0708,C,Second,child,False,NaN,Cherbourg,yes,False


In [3]:
df.describe()

,survived,pclass,age,sibsp,parch,fare
count,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    str     
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    str     
 8   class        891 non-null    category
 9   who          891 non-null    str     
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    str     
 13  alive        891 non-null    str     
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), str(5)
memory usage: 80.7 KB


In [5]:
df['deck'].unique()

[NaN, 'C', 'E', 'G', 'D', 'A', 'B', 'F']
Categories (7, str): ['A', 'B', 'C', 'D', 'E', 'F', 'G']

In [6]:
# assuming person is alive and finding the age of them
import datetime as dt
((dt.date.today().year - 1912) + df['age']).describe()

count    714.000000
mean     143.699118
std       14.526497
min      114.420000
25%      134.125000
50%      142.000000
75%      152.000000
max      194.000000
Name: age, dtype: float64

In [7]:
# Cheking whether the person who survived, is alive or not today - obviously accident happened in 1912 so no one is alive today.
def isalive(row):
    if row['survived'] == 1 and row['alive'] == "no":
        return row
    else:
        return
df1 = df.apply(isalive, axis=1)

In [8]:
if not df1.loc[0]:
    print("Nothing")

# 

Nothing


# analyze survival rates by class/gender/age

## class

In [9]:
df1 = pd.DataFrame(df.groupby('class')['survived'].count())
df1.rename(columns={'survived':'total'},inplace=True)
print(df1)

        total
class        
First     216
Second    184
Third     491


In [10]:
# Count and Survival rate by class
df1 = pd.DataFrame(df.groupby(['class','survived'])['alive'].count())


In [11]:
df3 = pd.DataFrame(index = df1.index.get_level_values(0), data=df1.values, columns = ["alive"])
print(df3)

        alive
class        
First      80
First     136
Second     97
Second     87
Third     372
Third     119


In [12]:
df1.index.get_level_values(0)

CategoricalIndex(['First', 'First', 'Second', 'Second', 'Third', 'Third'], categories=['First', 'Second', 'Third'], ordered=False, dtype='category', name='class')

In [13]:
df2 = pd.DataFrame(df1.groupby(level=0).sum()).rename(columns={"alive":"total"})
print(df2)

        total
class        
First     216
Second    184
Third     491


In [14]:
df3 = df3.join(df2)
df3

,alive,total
class,,
First,80,216
First,136,216
Second,97,184
Second,87,184
Third,372,491
Third,119,491


In [15]:
df3['rate'] = df3['alive']/df3['total'] * 100
df3

,alive,total,rate
class,,,
First,80,216,37.037037
First,136,216,62.962963
Second,97,184,52.717391
Second,87,184,47.282609
Third,372,491,75.763747
Third,119,491,24.236253


In [16]:
survival_rate_byClass = pd.DataFrame(df3.groupby('class').tail(n=1).round(decimals=2).rename(columns={"rate":"%survival_rate"}))
survival_rate_byClass

,alive,total,%survival_rate
class,,,
First,136,216,62.96
Second,87,184,47.28
Third,119,491,24.24


In [17]:
print(df[df['survived'] == 1].groupby('class')['survived'].count())

class
First     136
Second     87
Third     119
Name: survived, dtype: int64


In [18]:
df.shape[0]

891

In [19]:
grouped = pd.DataFrame(df.groupby(['class','survived'])['survived'].count())

In [20]:
# insight - Third class survival rate is very low and first class survival rate is high.

## gender

In [21]:
df1 = pd.DataFrame(df.groupby(['sex','survived'])['alive'].count())
df1

alive
sex    survived       
female 0            81
       1           233
male   0           468
       1           109

In [22]:
df2 = pd.DataFrame(index=df1.index.get_level_values(0), data=df1.values, columns=['alive'])
df2

,alive
sex,
female,81
female,233
male,468
male,109


In [23]:
df3 = pd.DataFrame(df1.groupby(level=0)['alive'].sum()).rename(columns={'alive':'total'})
df3

,total
sex,
female,314
male,577


In [24]:
survival_rate_bySex = pd.DataFrame(df2.join(df3)).groupby('sex').tail(n=1)
survival_rate_bySex['rate'] = (survival_rate_bySex['alive']/survival_rate_bySex['total']*100).round(decimals=2)
survival_rate_bySex.rename(columns={'rate':'%rate'})

,alive,total,%rate
sex,,,
female,233,314,74.20
male,109,577,18.89


In [25]:
survival_rate_bySex

,alive,total,rate
sex,,,
female,233,314,74.20
male,109,577,18.89


In [26]:
#insight - femail survival rate was 74% and male was only 19%

In [27]:
df.groupby(['sex','class','survived'])['alive'].count()

sex     class   survived
female  First   0             3
                1            91
        Second  0             6
                1            70
        Third   0            72
                1            72
male    First   0            77
                1            45
        Second  0            91
                1            17
        Third   0           300
                1            47
Name: alive, dtype: int64

## age

In [28]:
[age for age in df['age'] if age < 1]

[0.83, 0.92, 0.75, 0.75, 0.67, 0.42, 0.83]

In [29]:
df.groupby(['who','adult_male'])['alive'].count()

who    adult_male
child  False          83
man    True          537
woman  False         271
Name: alive, dtype: int64

In [30]:
# 0-5
# 6-12
# 13-18
# 19-25
# 26-40
# 41-60
# 60+
def age_group(age):
    if age<=5:
        return "0-5"
    elif age <=12:
        return "6-12"
    elif age<=18:
        return "13-18"
    elif age<=25:
        return "19-25"
    elif age<=40:
        return "26-40"
    elif age<=60:
        return "41-60"
    else:
        return "60+"
df['age_group(start-end)'] = df['age'].apply(age_group)
df['age_group(start-end)']

0      19-25
1      26-40
2      26-40
3      26-40
4      26-40
       ...  
886    26-40
887    19-25
888      60+
889    26-40
890    26-40
Name: age_group(start-end), Length: 891, dtype: str

In [31]:
df.groupby('age_group(start-end)')['alive'].count()

age_group(start-end)
0-5       44
13-18     70
19-25    162
26-40    263
41-60    128
6-12      25
60+      199
Name: alive, dtype: int64

In [32]:
df1 = pd.DataFrame(df.groupby(['age_group(start-end)','survived'])['alive'].count())
df1

alive
age_group(start-end) survived       
0-5                  0            13
                     1            31
13-18                0            40
                     1            30
19-25                0           108
                     1            54
26-40                0           152
                     1           111
41-60                0            78
                     1            50
6-12                 0            16
                     1             9
60+                  0           142
                     1            57

In [33]:
df2 = pd.DataFrame(df1.groupby('age_group(start-end)')['alive'].sum()).rename(columns={'alive':'total'})
df2

,total
age_group(start-end),
0-5,44
13-18,70
19-25,162
26-40,263
41-60,128
6-12,25
60+,199


In [34]:
df3 = pd.DataFrame(df1.groupby('age_group(start-end)')['alive'].tail(n=1)).droplevel(1)
df3

,alive
age_group(start-end),
0-5,31
13-18,30
19-25,54
26-40,111
41-60,50
6-12,9
60+,57


In [35]:
survival_rate_byAge = pd.DataFrame(df2.join(df3))
survival_rate_byAge['%rate'] = survival_rate_byAge['alive']/survival_rate_byAge['total']*100
survival_rate_byAge = pd.DataFrame(survival_rate_byAge.round(decimals=2).sort_index(key=lambda x: x.str.split(pat="-").str[0].str.replace('+',"").astype(int)))
survival_rate_byAge

,total,alive,%rate
age_group(start-end),,,
0-5,44,31,70.45
6-12,25,9,36.00
13-18,70,30,42.86
19-25,162,54,33.33
26-40,263,111,42.21
41-60,128,50,39.06
60+,199,57,28.64


In [36]:
#insight - childrens less than 5 years old has higher survival rate - which is good

# Dashboard